# Austin Metro Economic Productivity Map — Value per Acre

Urban3 / Strong-Towns style **value-per-acre** map: every parcel extruded by its economic value
per acre, so dense central land towers over big-box / suburban tracts.

**Metrics (per parcel):**
- `value_per_acre     = market_value / land_acres`  — what the land is worth
- `value_per_acre_adj = value_per_acre / pvs_ratio` — sales-ratio normalized for cross-county comparability
- `tax_per_acre       = taxable_value * effective_rate / land_acres` — fiscal layer (Travis only; others lack taxable_value)

**Data:** per-county ArcGIS FeatureServers (value+geometry+acreage bundled, no roll-join). Config +
fetcher in `v2_county_sources.py` / `v2_fetch_parcels.py`. Counties: Travis, Williamson, Hays. Verified 2026-06-29.

**Two scopes:**
- *Slice* (default): a downtown-Austin bbox over Travis — fast, for iteration + true-parcel 3D hero scene.
- *Metro*: all three counties (~638k parcels) loaded from a cached parquet; rendered as H3 hexes
  (2D folium choropleth + 3D extruded) because ~1M true polygons won't fit in a browser HTML.

Set `LOAD_METRO_CACHE = True` (after building the parquet) to switch to metro scope.

## Setup

In [1]:
import sys, math
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import pydeck as pdk
import folium
import h3

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
from v2_county_sources import COUNTY_SOURCES, PVS_RATIOS
from v2_fetch_parcels import fetch_county_gdf

OUT_DIR = REPO_ROOT / 'outputs'; OUT_DIR.mkdir(exist_ok=True)
PROC_DIR = REPO_ROOT / 'processed_data'; PROC_DIR.mkdir(exist_ok=True)
METRO_PARQUET = PROC_DIR / 'parcels_value_per_acre_metro.parquet'
print('counties:', list(COUNTY_SOURCES), '| pvs:', PVS_RATIOS)

counties: ['travis', 'williamson', 'hays'] | pvs: {'travis': 1.0, 'williamson': 0.96, 'hays': 0.97}


## Config

In [2]:
# --- scope ---
DOWNTOWN_BBOX = (-97.78, 30.24, -97.71, 30.30)   # central Austin hero area (lon/lat)
SCOPE = {'travis': DOWNTOWN_BBOX}                 # {county: bbox-or-None}; used for live fetch
LOAD_METRO_CACHE = False                          # True -> load full metro from METRO_PARQUET instead

# --- cleaning ---
MIN_ACRES = 0.01          # drop sub-436-sqft error slivers (absurd $/acre artifacts)
EFFECTIVE_TAX_RATE = 0.021  # blended city+county+school+special districts (~2.1%); approx for tax/acre

# --- rendering ---
MAX_HEIGHT = 2500.0       # tallest (p99) parcel/hex extrudes to this many meters; rest scales linearly
ELEV_METRIC = 'value_per_acre'   # or 'value_per_acre_adj' (better cross-county) or 'tax_per_acre'
SIMPLIFY_M = 2.0          # geometry simplification tolerance in meters (0 = off) for the 3D parcel scene
H3_RES = 8               # H3 resolution for metro aggregation (8 ~0.7 km2/hex; 9 ~0.1 km2)

## Fetch (or load cached metro)

In [3]:
if LOAD_METRO_CACHE:
    print('loading metro parquet:', METRO_PARQUET)
    g = gpd.read_parquet(METRO_PARQUET)   # already cleaned + metrics computed
    PRECOMPUTED = True
else:
    frames = []
    for county, bbox in SCOPE.items():
        print(f'fetching {county}' + (f' bbox {bbox}' if bbox else ' (full county)') + ' ...')
        gc = fetch_county_gdf(county, bbox=bbox, only_valued=True, with_geometry=True)
        print(f'  {len(gc):,} parcels'); frames.append(gc)
    g = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs='EPSG:4326')
    PRECOMPUTED = False
print(f'{len(g):,} parcels')

fetching travis bbox (-97.78, 30.24, -97.71, 30.3) ...


  21,092 parcels
21,092 parcels


## Clean + compute metrics
Drop null geometry, sub-`MIN_ACRES` slivers, and zero/exempt value; compute the three metrics.
(Skipped automatically when loading the cached parquet, which is already clean.)

In [4]:
if not PRECOMPUTED:
    g = g[g.geometry.notna()].copy()
    g['land_acres'] = pd.to_numeric(g['land_acres'], errors='coerce')
    g['market_value'] = pd.to_numeric(g['market_value'], errors='coerce')
    before = len(g)
    g = g[(g['land_acres'] >= MIN_ACRES) & (g['market_value'] > 0)]
    print(f'dropped {before - len(g):,} parcels below {MIN_ACRES} ac or zero value')
    g['value_per_acre'] = g['market_value'] / g['land_acres']
    g['pvs_ratio'] = g['county'].map(PVS_RATIOS)
    g['value_per_acre_adj'] = g['value_per_acre'] / g['pvs_ratio']
    if 'taxable_value' in g.columns:
        g['taxable_value'] = pd.to_numeric(g['taxable_value'], errors='coerce')
        g['tax_per_acre'] = g['taxable_value'] * EFFECTIVE_TAX_RATE / g['land_acres']
    else:
        g['tax_per_acre'] = np.nan
    g = g[g['value_per_acre'].between(1, 1e12)]
print(f'{len(g):,} parcels after cleaning')
g[['value_per_acre', 'value_per_acre_adj', 'tax_per_acre']].describe()

dropped 9 parcels below 0.01 ac or zero value
21,083 parcels after cleaning


,value_per_acre,value_per_acre_adj,tax_per_acre
count,2.108300e+04,2.108300e+04,2.108200e+04
mean,7.644454e+06,7.644454e+06,1.605347e+05
std,1.428617e+07,1.428617e+07,3.000166e+05
min,1.296092e+01,1.296092e+01,2.721794e-01
25%,3.876747e+06,3.876747e+06,8.141116e+04
50%,5.439956e+06,5.439956e+06,1.142369e+05
75%,8.065518e+06,8.065518e+06,1.693817e+05
max,4.836484e+08,4.836484e+08,1.015662e+07


## Validation — the Urban3 sanity story
Top by value/acre = dense central commercial/residential; bottom = large low-value tracts & fringe land.

In [5]:
cols = [c for c in ['county','parcel_id','land_use','market_value','land_acres','value_per_acre'] if c in g.columns]
print('TOP 10 by value/acre'); display(g.sort_values('value_per_acre', ascending=False)[cols].head(10))
print('BOTTOM 10 by value/acre'); display(g.sort_values('value_per_acre')[cols].head(10))

TOP 10 by value/acre


,county,parcel_id,land_use,market_value,land_acres,value_per_acre
4970,travis,0206011606,OFF HI-RISE >= 6,196119412,0.4055,4.836484e+08
19202,travis,0206011205,LUXURY HI-RISE APTS 100+,192780000,0.4055,4.754131e+08
5009,travis,0205021001,HIRISE CONDO/APT,188356279,0.4055,4.645038e+08
20556,travis,0203031001,LUXURY HI-RISE APTS 100+,233950000,0.5405,4.328400e+08
20013,travis,0206011901,OFF HI-RISE >= 6,387556329,0.9377,4.133052e+08
20249,travis,0206030709,HOTEL-FULL SERVC,98000000,0.2535,3.865878e+08
14965,travis,0206030816,HOTEL-FULL SERVC,58000000,0.1593,3.640929e+08
3277,travis,0214011303,APARTMENT 100+,143490000,0.4014,3.574738e+08
1512,travis,0210021714,OFF HI-RISE >= 6,147991225,0.4406,3.358857e+08
12150,travis,0105001001,OFF HI-RISE >= 6,275633332,0.8230,3.349129e+08


BOTTOM 10 by value/acre


,county,parcel_id,land_use,market_value,land_acres,value_per_acre
17112,travis,0104090221,NaN,20,1.5431,12.960923
18099,travis,0217130103,NaN,20000,30.1221,663.964332
18103,travis,0219120242,NaN,10000,13.1408,760.988676
14167,travis,0215080169,NaN,210,0.0483,4347.826087
10539,travis,NaN,RETAIL STORE,750000,163.6395,4583.245488
20589,travis,0105000103,NaN,750000,163.6395,4583.245488
16410,travis,0214000102,NaN,1500,0.3000,5000.000000
20971,travis,0405061601,NaN,26250,3.9209,6694.891479
10934,travis,0404070326,NaN,5478,0.6130,8936.378467
16176,travis,0404070615,Detail Only,7699,0.7708,9988.323819


## Shared color ramp (log-scaled blue → yellow → red)

In [6]:
def make_lognorm(series, plo=0.01, phi=0.99):
    p_lo = max(series.quantile(plo), 1.0); p_hi = series.quantile(phi)
    lo, hi = math.log(p_lo), math.log(p_hi)
    def norm(v):
        x = (math.log(max(v, 1)) - lo) / (hi - lo)
        return min(max(x, 0.0), 1.0)
    return norm, p_lo, p_hi

STOPS = [(0.0, (30, 60, 160)), (0.5, (240, 220, 60)), (1.0, (200, 30, 30))]
def ramp(t):
    for i in range(len(STOPS) - 1):
        t0, c0 = STOPS[i]; t1, c1 = STOPS[i + 1]
        if t <= t1:
            f = (t - t0) / (t1 - t0) if t1 > t0 else 0
            return [int(c0[j] + f * (c1[j] - c0[j])) for j in range(3)]
    return list(STOPS[-1][1])
def ramp_hex(t):
    return '#%02x%02x%02x' % tuple(ramp(t))

## 3D true-parcel scene (slice / bounded scope)
pydeck `PolygonLayer`, extruded + colored by `ELEV_METRIC`. Best for the slice — true parcels at full
metro scale produce a browser-breaking HTML, so use the H3 layers below for metro.

In [7]:
POLY_LIMIT = 200_000
if len(g) > POLY_LIMIT:
    print(f'{len(g):,} parcels > {POLY_LIMIT:,}: skipping true-parcel 3D (use H3 metro scene). '
          'Narrow SCOPE to a bbox for this layer.')
    out_html_3d = None
else:
    gg = g.copy()
    if SIMPLIFY_M > 0:
        gg['geometry'] = gg.to_crs(2277).geometry.simplify(SIMPLIFY_M).to_crs(4326)
    norm, p_lo, p_hi = make_lognorm(gg[ELEV_METRIC])
    elev_scale = MAX_HEIGHT / p_hi
    print(f'{ELEV_METRIC}: p01={p_lo:,.0f} p99={p_hi:,.0f} (tallest -> {MAX_HEIGHT:.0f} m)')
    def rings(geom):
        if geom.geom_type == 'Polygon': return [list(geom.exterior.coords)]
        if geom.geom_type == 'MultiPolygon': return [list(p.exterior.coords) for p in geom.geoms]
        return []
    poly_data = []
    for _, r in gg.iterrows():
        v = r[ELEV_METRIC]; color = ramp(norm(v)); v_clip = min(v, p_hi)
        for ring in rings(r.geometry):
            poly_data.append({'polygon': [[x, y] for x, y in ring],
                'elevation': v_clip * elev_scale, 'color': color,
                'vpa': f"${r['value_per_acre']:,.0f}/acre", 'mkt': f"${r['market_value']:,.0f}",
                'acres': f"{r['land_acres']:.3f}"})
    cx, cy = float(gg.geometry.union_all().centroid.x), float(gg.geometry.union_all().centroid.y)
    layer = pdk.Layer('PolygonLayer', poly_data, get_polygon='polygon', get_elevation='elevation',
        get_fill_color='color', extruded=True, pickable=True, elevation_scale=1)
    deck = pdk.Deck(layers=[layer],
        initial_view_state=pdk.ViewState(latitude=cy, longitude=cx, zoom=13, pitch=55, bearing=20),
        map_style='dark', tooltip={'text': '{vpa}\nMarket: {mkt}\nAcres: {acres}'})
    out_html_3d = OUT_DIR / 'map_value_per_acre_3d.html'
    deck.to_html(str(out_html_3d), notebook_display=False)
    print(f'{len(poly_data):,} rings -> {out_html_3d}')

value_per_acre: p01=956,781 p99=42,804,769 (tallest -> 2500 m)


21,223 rings -> /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_3d.html


## H3 hex aggregation (metro-scale)
Bin parcels to H3 cells by centroid; per hex, `value_per_acre = sum(market_value) / sum(land_acres)`.
This is the scalable representation for the full metro — used by both the 2D and 3D maps below.

In [8]:
cent = g.geometry.centroid
hexes = [h3.latlng_to_cell(y, x, H3_RES) for x, y in zip(cent.x.values, cent.y.values)]
agg = pd.DataFrame({'hex': hexes, 'market_value': g['market_value'].values,
                    'land_acres': g['land_acres'].values})
hx = agg.groupby('hex').agg(market_value=('market_value','sum'),
                            land_acres=('land_acres','sum'),
                            n=('hex','size')).reset_index()
hx['value_per_acre'] = hx['market_value'] / hx['land_acres']
print(f'{len(hx):,} H3 cells (res {H3_RES}) covering {len(g):,} parcels')
hx['value_per_acre'].describe()

75 H3 cells (res 8) covering 21,083 parcels


/var/folders/fs/q5d1pm490h17_b7dlph23s8c0000gn/T/ipykernel_98417/4025811405.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = g.geometry.centroid


count    7.500000e+01
mean     7.664406e+06
std      8.091287e+06
min      4.583245e+03
25%      4.075881e+06
50%      5.374669e+06
75%      7.090706e+06
max      4.303018e+07
Name: value_per_acre, dtype: float64

### 2D shareable map (folium choropleth)

In [9]:
norm_h, p_lo_h, p_hi_h = make_lognorm(hx['value_per_acre'])
clat = float(g.geometry.centroid.y.mean()); clon = float(g.geometry.centroid.x.mean())
m = folium.Map(location=[clat, clon], zoom_start=11, tiles='CartoDB dark_matter')
for _, r in hx.iterrows():
    boundary = h3.cell_to_boundary(r['hex'])  # [(lat, lng), ...]
    folium.Polygon(locations=[[lat, lng] for lat, lng in boundary],
        color=None, fill=True, fill_color=ramp_hex(norm_h(r['value_per_acre'])),
        fill_opacity=0.75, weight=0,
        tooltip=f"${r['value_per_acre']:,.0f}/acre · {int(r['n'])} parcels").add_to(m)
out_html_2d = OUT_DIR / 'map_value_per_acre_metro.html'
m.save(str(out_html_2d))
print('wrote', out_html_2d)

wrote /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_metro.html


/var/folders/fs/q5d1pm490h17_b7dlph23s8c0000gn/T/ipykernel_98417/3339028288.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  clat = float(g.geometry.centroid.y.mean()); clon = float(g.geometry.centroid.x.mean())


### 3D extruded hexes (metro hero scene)

In [10]:
elev_scale_h = MAX_HEIGHT / p_hi_h
hx3 = hx.copy()
hx3['elevation'] = hx3['value_per_acre'].clip(upper=p_hi_h) * elev_scale_h
hx3['color'] = hx3['value_per_acre'].apply(lambda v: ramp(norm_h(v)))
hx3['vpa'] = hx3['value_per_acre'].apply(lambda v: f'${v:,.0f}/acre')
layer_h = pdk.Layer('H3HexagonLayer', hx3, get_hexagon='hex', get_elevation='elevation',
    get_fill_color='color', extruded=True, pickable=True, elevation_scale=1, coverage=1)
deck_h = pdk.Deck(layers=[layer_h],
    initial_view_state=pdk.ViewState(latitude=clat, longitude=clon, zoom=10, pitch=50, bearing=15),
    map_style='dark', tooltip={'text': '{vpa}'})
out_html_metro_3d = OUT_DIR / 'map_value_per_acre_metro_3d.html'
deck_h.to_html(str(out_html_metro_3d), notebook_display=False)
print('wrote', out_html_metro_3d)

wrote /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_metro_3d.html


## Outputs
- `outputs/map_value_per_acre_3d.html` — true-parcel 3D scene (slice scope)
- `outputs/map_value_per_acre_metro.html` — 2D H3 choropleth (shareable)
- `outputs/map_value_per_acre_metro_3d.html` — 3D extruded H3 hexes (metro hero)

To render the full metro: build the parquet (`processed_data/parcels_value_per_acre_metro.parquet`)
then set `LOAD_METRO_CACHE = True` and re-run. Switch `ELEV_METRIC = 'value_per_acre_adj'` for the
cross-county-comparable layer that softens the county-line seam.